# 00_b1 — Austria: municipality population by age

Part of the `00_b*` series that replaces the five legacy population tables in the local
`dbis` Postgres (read by `traintran/population.py` / `harmonize_population_lau2.ipynb`) with
newer national-statistics files, each brought to the **same age-band widths** and keyed on
`LAU CODE` from `EU-27-LAU-2025-NUTS-2024.xlsx`.

> **Working notebook — not for publication.** It connects to the local DB only to take the legacy
> table as a baseline; once the harmonised output exists, nothing downstream needs the DB.

| | dataset | location |
|---|---|---|
| legacy (DB) | `public.atpopulation` | local `dbis` Postgres |
| new (2025) | Statistik Austria open data `OGD_bevstandjbab2002_BevStand_2026` — *Bevölkerungsstand zu Jahresbeginn 2026* (population on 1 Jan 2026, territorial status 2026), by municipality × single year of age × sex, CC BY 4.0 (<https://data.statistik.gv.at/web/meta.jsp?dataset=OGD_bevstandjbab2002_BevStand_2026>) | `data/eu/ref-national-statistical-offices/at/ogd/` |

**Legacy notes.** Mixed-level table: country, Bundesländer, Bezirke and municipalities in one. Only `popTotal`, `pctUnder15`, `pctOver65` — no counts per age band. Join: `id` ≥ 10000, plus the 15 Statutarstädte recovered from their Bezirk id (`id*100+1`) and Wien (`9` → `90001`).

In [1]:
import os, sys

# Run from the project root so every relative path ('data/...', 'traintran/...') resolves the
# same whether the notebook is launched from notebooks/ or from the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: e:\notes\api_zssk\project


In [2]:
import os
from pathlib import Path

import pandas as pd
# import sqlalchemy as sa

import traintran  # noqa: F401  (fixes PROJ on import)
from traintran import population as pop
from traintran.config import default_config

COUNTRY = 'AT'
OUT_DIR = Path('data/eu/ref-national-statistical-offices') / COUNTRY.lower()
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = default_config()
# Database comparison disabled. The 2025 population comes from the statistical office's files below; the
# database held only the older 2021 tables used for a comparison that feeds no output. To re-enable it, set
# TRAINTRAN_PG_URL in the environment and uncomment these lines and the cells marked '[disabled: ...]'.
# engine = sa.create_engine(os.environ['TRAINTRAN_PG_URL'])
print('Country   :', COUNTRY)
print('Output dir:', OUT_DIR)
# print('Postgres  :', cfg.pg_url.rsplit('@', 1)[-1])

Country   : AT
Output dir: data\eu\ref-national-statistical-offices\at


## 1. Legacy — local DB

The raw table(s) as they sit in `dbis`, geometry columns dropped. A parquet snapshot is written
next to the new source (skip-if-present) so the comparison in §3 can later run without the DB.

In [3]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# GEOM_COLS = {'geom', 'geojson', 'wkb_geometry', 'geometry'}
# LEGACY_TABLES = ['public.atpopulation']


# def read_table(qualified):
#     schema, table = qualified.split('.', 1)
#     table = table.strip('"')
#     cols = [c['name'] for c in sa.inspect(engine).get_columns(table, schema=schema)
#             if c['name'].lower() not in GEOM_COLS]
#     sel = ', '.join('"' + c.replace('"', '""') + '"' for c in cols)
#     return pd.read_sql(f'select {sel} from {schema}."{table}"', engine)


# legacy_raw = {}
# for t in LEGACY_TABLES:
#     legacy_raw[t] = df = read_table(t)
#     print(f'{t}: {len(df):,} rows x {df.shape[1]} cols')
#     print('  ', list(df.columns))

In [4]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# for t, df in legacy_raw.items():
#     stem = t.replace('.', '_').replace(chr(34), '')
#     snap = OUT_DIR / f'legacy_db_{stem}.parquet'
#     if snap.exists():
#         print('[exists]', snap)
#     else:
#         df.to_parquet(snap, index=False)
#         print('[wrote] ', snap, f'({len(df):,} rows)')

### 1a. Baseline — the current loader on the legacy table

`pop.LOADERS['AT']` is exactly what `harmonize_population_lau2.ipynb` runs today; its keyed and
banded result is what the 2025 data has to reproduce in structure (same `lau_code` coverage,
same harmonised bands).

In [5]:
lau = pop.load_workbook_lau(cfg.lau_file, countries=(COUNTRY,))
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# legacy = pop.harmonise_bands(pop.LOADERS[COUNTRY](engine, lau), COUNTRY)
# legacy['country'] = COUNTRY
# legacy['ref_year'] = 2021

# print(pop.match_report({COUNTRY: legacy}, lau).to_string(index=False))
# print()
# print('band_method  :', legacy.band_method.unique().tolist())
# print('band_complete:', int(legacy.band_complete.sum()), '/', len(legacy))
# print('pop_total    :', f'{legacy.pop_total.sum():,.0f}')
# print()
# print(legacy.head().to_string())

## 2. New (2025) — Statistik Austria open data, `OGD_bevstandjbab2002_BevStand_2026`

*Bevölkerungsstand zu Jahresbeginn 2026* — population on **1 Jan 2026** (territorial status 2026)
by municipality, single year of age and sex, CC BY 4.0, from the open-data portal
(<https://data.statistik.gv.at/web/meta.jsp?dataset=OGD_bevstandjbab2002_BevStand_2026>). 1 Jan 2026 is the stock at the end of 2025, i.e. the same reference point as CZ, PL,
SK and HU (31 Dec 2025); carried as `ref_year = 2025`, `ref_date = 2026-01-01`.

It replaces `AEST_MunicipalityResults_2024.ods` (31 Oct 2024), which published only the
percentages under 15 / 65+, so its bands had to be reconstructed. Here the bands are **exact counts**.

**Files.** `OGD_bevstandjbab2002_BevStand_2026.csv` holds codes only; the column names are in `…_HEADER.csv` and every coded
column has its own code list `…_<column>.csv`:

| column | meaning | example code → value |
|---|---|---|
| `C-A10-0` | year | `A10-2026` → 2026 |
| `C-C11-0` | sex | `C11-1` male, `C11-2` female |
| `C-GRGEMAKT-0` | municipality | `GRGEMAKT-10101` → Eisenstadt, GKZ `10101` = LAU code |
| `C-GALTEJ112-0` | age in single years | `GALTEJ112-<k>` = age **k − 1**; 100+ is `GALT5J100-21` |
| `F-ISIS-1` | count (persons) | — |

In [6]:
import requests

OGD = 'OGD_bevstandjbab2002_BevStand_2026'
OGD_URL = 'https://data.statistik.gv.at/data/OGD_bevstandjbab2002_BevStand_2026'
OGD_DIR = OUT_DIR / 'ogd'
OGD_DIR.mkdir(parents=True, exist_ok=True)
OGD_PARTS = ['', '_HEADER', '_C-A10-0', '_C-C11-0', '_C-GRGEMAKT-0', '_C-GALTEJ112-0']

for part in OGD_PARTS:
    p = OGD_DIR / f'{OGD}{part}.csv'
    if p.exists():
        print('[exists]', p.name)
        continue
    r = requests.get(f'{OGD_URL}{part}.csv', timeout=300)
    r.raise_for_status()
    p.write_bytes(r.content)
    print('[wrote] ', p.name, f'({len(r.content) / 1e6:.1f} MB)')
NEW_FILE = OGD_DIR / f'{OGD}.csv'

[exists] OGD_bevstandjbab2002_BevStand_2026.csv
[exists] OGD_bevstandjbab2002_BevStand_2026_HEADER.csv
[exists] OGD_bevstandjbab2002_BevStand_2026_C-A10-0.csv
[exists] OGD_bevstandjbab2002_BevStand_2026_C-C11-0.csv
[exists] OGD_bevstandjbab2002_BevStand_2026_C-GRGEMAKT-0.csv
[exists] OGD_bevstandjbab2002_BevStand_2026_C-GALTEJ112-0.csv


### 2a. Raw data and code lists

Every code in the data must appear in its code list (asserted), so a re-issued file with new
codes fails here instead of silently dropping rows.

In [7]:
def code_list(col):
    c = pd.read_csv(OGD_DIR / f'{OGD}_{col}.csv', sep=';', dtype=str)
    return c.set_index('code')['name']


header = pd.read_csv(OGD_DIR / f'{OGD}_HEADER.csv', sep=';', dtype=str, usecols=[0, 1, 2])
print(header.to_string(index=False), '\n')

at_raw = pd.read_csv(NEW_FILE, sep=';', dtype=str)
lists = {c: code_list(c) for c in ['C-A10-0', 'C-C11-0', 'C-GRGEMAKT-0', 'C-GALTEJ112-0']}
for c, cl in lists.items():
    unknown = set(at_raw[c]) - set(cl.index)
    assert not unknown, f'{c}: codes not in the code list: {sorted(unknown)[:5]}'
    print(f'{c:<15} {at_raw[c].nunique():>5} codes used of {len(cl):>5} listed')
assert at_raw['C-A10-0'].unique().tolist() == ['A10-2026'], at_raw['C-A10-0'].unique()
print(f'\n{len(at_raw):,} rows')
at_raw.head()

         code                                            name                                     en_name
     F-ISIS-1                                        Fallzahl                                      Number
      C-A10-0                                            Jahr                                Time section
      C-C11-0                                      Geschlecht                                         Sex
 C-GRGEMAKT-0 Gemeinde (Vergröberung über Politischen Bezirk) Commune (aggregation by political district)
C-GALTEJ112-0                           Alter in Einzeljahren                         Age in single years 

C-A10-0             1 codes used of     1 listed
C-C11-0             2 codes used of     2 listed
C-GRGEMAKT-0     2114 codes used of  2116 listed
C-GALTEJ112-0     101 codes used of   101 listed

393,942 rows


,C-A10-0,C-C11-0,C-GRGEMAKT-0,C-GALTEJ112-0,F-ISIS-1
0,A10-2026,C11-1,GRGEMAKT-10101,GALTEJ112-1,63
1,A10-2026,C11-1,GRGEMAKT-10101,GALTEJ112-2,66
2,A10-2026,C11-1,GRGEMAKT-10101,GALTEJ112-3,65
3,A10-2026,C11-1,GRGEMAKT-10101,GALTEJ112-4,64
4,A10-2026,C11-1,GRGEMAKT-10101,GALTEJ112-5,76


### 2b. Harmonise — one row per municipality

Ages are decoded from the code list label (`'0 Jahre'`, …, `'100 Jahre und älter'`), so the
off-by-one in the code numbers cannot bite. Both sexes are summed, then single years go to
`popAge0to14` / `popAge15to64` / `popAge65over`. `name` and `id` (GKZ) follow the legacy table.

**Wien** is published as its 23 Bezirke (`90101` … `92301`); the LAU workbook has Wien as one
municipality, `90001`. The districts are summed into one `90001` row (kept separately in
`at_wien_districts`); `includes_gkz` lists them.

In [8]:
age_label = lists['C-GALTEJ112-0']
age = age_label.str.extract(r'^(\d+)')[0].astype(int)
assert age.sort_values().tolist() == list(range(101)), 'ages are not 0..100+'

x = at_raw.assign(
    id=at_raw['C-GRGEMAKT-0'].str.split('-').str[1],
    age=at_raw['C-GALTEJ112-0'].map(age),
    n=pd.to_numeric(at_raw['F-ISIS-1'], errors='raise'))
assert x['n'].ge(0).all() and (x['n'] % 1 == 0).all()
x['band'] = pd.cut(x['age'], [-1, 14, 64, 200], labels=['popAge0to14', 'popAge15to64', 'popAge65over'])
B = ['popAge0to14', 'popAge15to64', 'popAge65over']

w = x.pivot_table(index='id', columns='band', values='n', aggfunc='sum', observed=False)[B].fillna(0)
w['popTotal'] = x.groupby('id')['n'].sum()
names = lists['C-GRGEMAKT-0'].rename(index=lambda c: c.split('-')[1])
w['name'] = names.reindex(w.index).str.replace(r'\s*<\d+>$', '', regex=True)
w = w.reset_index()[['id', 'name', 'popTotal', *B]]

WIEN_LAU = '90001'
is_wien = w['id'].str.match(r'^9\d{2}01$') & (w['id'] != WIEN_LAU)
at_wien_districts = w[is_wien].copy()
assert len(at_wien_districts) == 23, len(at_wien_districts)
wien = at_wien_districts[['popTotal', *B]].sum().to_frame().T
wien.insert(0, 'name', 'Wien')
wien.insert(0, 'id', WIEN_LAU)
wien['includes_gkz'] = ';'.join(sorted(at_wien_districts['id']))
at_new = pd.concat([w[~is_wien], wien], ignore_index=True)
at_new = at_new.astype({c: 'int64' for c in ['popTotal', *B]})

assert at_new['id'].is_unique
assert (at_new[B].sum(axis=1) == at_new['popTotal']).all(), 'bands do not add up'
assert at_new['popTotal'].sum() == x['n'].sum(), 'population lost'
at_new['ref_year'] = 2025
at_new['ref_date'] = '2026-01-01'

print(f'at_new: {len(at_new):,} municipalities (23 Wien Bezirke -> 1 row {WIEN_LAU})')
print(f'population total: {at_new.popTotal.sum():,}   Wien {wien.popTotal.item():,}')
print('bands sum to popTotal: all rows')
at_new.head()

at_new: 2,092 municipalities (23 Wien Bezirke -> 1 row 90001)
population total: 9,215,956   Wien 2,040,914
bands sum to popTotal: all rows


band,id,name,popTotal,popAge0to14,popAge15to64,popAge65over,includes_gkz,ref_year,ref_date
0,10101,Eisenstadt,16396,2238,10582,3576,NaN,2025,2026-01-01
1,10201,Rust,1918,212,1137,569,NaN,2025,2026-01-01
2,10301,Breitenbrunn am Neusiedler See,1931,218,1072,641,NaN,2025,2026-01-01
3,10302,Donnerskirchen,1856,212,1152,492,NaN,2025,2026-01-01
4,10303,Großhöflein,2078,279,1336,463,NaN,2025,2026-01-01


### 2c. Joinability — new GKZ vs the 2025 workbook, and vs the legacy table

The GKZ is the LAU code, so the new table joins directly. The legacy table needed the
Statutarstadt and Wien recovery rules (`pop.load_at`); its LAU-keyed result (§1a) is compared
on `lau_code`.

In [9]:
at_new_lau = at_new.assign(lau_code=at_new['id'])
print(pop.match_report({COUNTRY: at_new_lau}, lau).to_string(index=False))

wb = set(lau['lau_code'])
print('\nnew LAU codes not in the 2025 workbook:')
print(at_new_lau.loc[~at_new_lau.lau_code.isin(wb), ['lau_code', 'name', 'popTotal']].to_string(index=False) or '  none')
print('\nworkbook LAU codes not covered by the new file:')
print(lau.loc[~lau.lau_code.isin(at_new_lau.lau_code), ['lau_code', 'lau_name_national']].to_string(index=False) or '  none')

nm = at_new_lau.merge(lau[['lau_code', 'lau_name_national']], on='lau_code')
renamed = nm[nm['name'].str.strip() != nm['lau_name_national'].str.strip()]
print(f'\nsame code, different name: {len(renamed)}')
print(renamed[['lau_code', 'name', 'lau_name_national']].head(10).to_string(index=False))

country  source_rows  workbook_lau  matched  source_unmatched  lau_uncovered
     AT         2092          2093     2091                 1              2

new LAU codes not in the 2025 workbook:
lau_code        name  popTotal
   62280 Fürstenfeld     10358

workbook LAU codes not covered by the new file:
lau_code lau_name_national
   62252            Söchau
   62267       Fürstenfeld

same code, different name: 1
lau_code              name lau_name_national
   41346 St. Stefan Afiesl St. Stefan-Afiesl


### 2d. Territorial changes — merged municipalities kept as one row

Between the workbook (2025) and territorial status 2026, Söchau (62252) and Fürstenfeld (62267)
merged into **Fürstenfeld (62280)**. It stays one row under its new code; `replaces_lau` lists the
workbook codes it covers. Any other unmatched code fails the assert.

In [10]:
# new LAU code -> workbook (2025) LAU codes it replaces
AT_MERGERS = {'62280': ['62252', '62267']}

at_new_lau['replaces_lau'] = at_new_lau['lau_code'].map(lambda c: ';'.join(AT_MERGERS.get(c, [])) or None)
covered = set(at_new_lau['lau_code']) | {o for olds in AT_MERGERS.values() for o in olds}
unexplained_new = sorted(set(at_new_lau['lau_code']) - wb - set(AT_MERGERS))
unexplained_wb = sorted(wb - covered)
assert not unexplained_new and not unexplained_wb, (unexplained_new, unexplained_wb)

print(f'new rows            : {len(at_new_lau):,}')
print(f'workbook LAU covered: {len(wb & covered):,} / {len(wb):,}  ({sum(map(len, AT_MERGERS.values()))} via {len(AT_MERGERS)} merger, Wien via its Bezirke)')
print(at_new_lau.loc[at_new_lau.replaces_lau.notna(), ['lau_code', 'name', 'popTotal', 'replaces_lau']].to_string(index=False))
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# old_parts = legacy.set_index('lau_code')['pop_total']
# for new_code, olds in AT_MERGERS.items():
#     before = old_parts.reindex(olds).sum()
#     after = at_new_lau.loc[at_new_lau.lau_code == new_code, 'popTotal'].item()
#     print(f'{new_code}: legacy {" + ".join(olds)} = {before:,.0f}  ->  new {after:,}  ({100 * (after - before) / before:+.1f}%)')

new rows            : 2,092
workbook LAU covered: 2,093 / 2,093  (2 via 1 merger, Wien via its Bezirke)
lau_code        name  popTotal replaces_lau
   62280 Fürstenfeld     10358  62252;62267


In [11]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# # Same LAU, two vintages: legacy (pct-derived bands) -> new (exact counts).
# old_by_lau = legacy.set_index('lau_code')
# old_by_lau = old_by_lau.rename(index={o: n for n, olds in AT_MERGERS.items() for o in olds})
# old_by_lau = old_by_lau.groupby(level=0)[['pop_total', 'y_0_14', 'y_15_64', 'y_65plus']].sum()
# cmp = old_by_lau.reset_index().merge(at_new_lau[['lau_code', 'name', 'popTotal', *B]], on='lau_code',
#                                      how='outer', indicator=True)
# print(cmp['_merge'].value_counts().to_string())
# b = cmp[cmp._merge == 'both'].copy()
# b['diff_%'] = 100 * (b.popTotal - b.pop_total) / b.pop_total
# print(f'\ntotal  legacy {b.pop_total.sum():,.0f}  ->  new {b.popTotal.sum():,.0f}')
# print(b['diff_%'].describe().round(2).to_string())
# print('\nband shares (%), legacy (derived from %) -> new (exact):')
# for o, n in [('y_0_14', 'popAge0to14'), ('y_15_64', 'popAge15to64'), ('y_65plus', 'popAge65over')]:
#     print(f'  {o:<9} {100 * b[o].sum() / b.pop_total.sum():5.1f} -> {100 * b[n].sum() / b.popTotal.sum():5.1f}')
# print('\nlargest relative changes:')
# print(b.reindex(b['diff_%'].abs().sort_values(ascending=False).index)
#        .head(10)[['lau_code', 'name', 'pop_total', 'popTotal', 'diff_%']].round(1).to_string(index=False))

### 2e. Join to the v4atua LAU subset (GeoJSON)

Austrian features only (`CNTR_CODE = 'AT'`), key `GISCO_ID = 'AT_<LAU code>'`. Like the workbook
it predates the Söchau + Fürstenfeld merger, so the two polygons are dissolved into one 62280 row
(population and area summed). Where the parts have different DEGURBA classes the merged row takes
the class of its **most populous part**: Söchau is rural (3), Fürstenfeld a town (2) with 86% of
the population, so 62280 is 2. Wien is a single polygon (`AT_90001`) in the GeoJSON already.

In [12]:
import geopandas as gpd

LAU_SUBSET = Path('data/eu/ref-lau-2024-01m/subsetv4atua/LAU_RG_01M_2024_4326.geojson')
at_geo = gpd.read_file(LAU_SUBSET, where="CNTR_CODE = 'AT'")
at_geo['lau_code'] = at_geo['GISCO_ID'].str.split('_', n=1).str[1]
assert at_geo['lau_code'].is_unique
print(f'{LAU_SUBSET.name}: {len(at_geo):,} AT features')

old_to_new = {o: n for n, olds in AT_MERGERS.items() for o in olds}
at_geo['lau_code_2026'] = at_geo['lau_code'].replace(old_to_new)
agg = at_geo.dissolve(by='lau_code_2026', aggfunc={
    'GISCO_ID': lambda s: ';'.join(sorted(s)), 'LAU_NAME': lambda s: ' + '.join(sorted(s)),
    'POP_2024': 'sum', 'AREA_KM2': 'sum'}).reset_index()
top = at_geo.sort_values('POP_2024', ascending=False).drop_duplicates('lau_code_2026')
agg['DGURBA'] = agg['lau_code_2026'].map(top.set_index('lau_code_2026')['DGURBA'])
print(f'after folding {len(old_to_new)} predecessors: {len(agg):,} features')

jg = agg.merge(at_new_lau[['lau_code', 'name', 'popTotal', *B, 'replaces_lau']],
               left_on='lau_code_2026', right_on='lau_code', how='outer', indicator=True)
print('\n' + jg['_merge'].value_counts().to_string())
print('\ngeojson only:', jg.loc[jg._merge == 'left_only', 'lau_code_2026'].tolist())
print('new file only:', jg.loc[jg._merge == 'right_only', 'lau_code'].tolist())
print('\nmerged row and Wien:')
print(jg.loc[jg.replaces_lau.notna() | (jg.lau_code == WIEN_LAU),
             ['lau_code_2026', 'GISCO_ID', 'LAU_NAME', 'name', 'POP_2024', 'popTotal', 'DGURBA']].to_string(index=False))

m = jg[jg._merge == 'both'].copy()
m['diff_%'] = 100 * (m.popTotal - m.POP_2024) / m.POP_2024
print(f'\npopulation  GISCO POP_2024 {m.POP_2024.sum():,.0f}  vs  new popTotal {m.popTotal.sum():,.0f}')
print(m['diff_%'].describe().round(2).to_string())
name_diff = m[m.LAU_NAME.str.strip() != m.name.str.strip()]
print(f'\nsame code, different name: {len(name_diff)}')
print(name_diff[['lau_code_2026', 'LAU_NAME', 'name']].head(10).to_string(index=False))

LAU_RG_01M_2024_4326.geojson: 2,092 AT features
after folding 2 predecessors: 2,092 features

_merge
both          2092
left_only        0
right_only       0

geojson only: []
new file only: []

merged row and Wien:
lau_code_2026 GISCO_ID    LAU_NAME        name  POP_2024  popTotal  DGURBA
        62280 AT_62280 Fürstenfeld Fürstenfeld   10368.0     10358       2
        90001 AT_90001        Wien        Wien 2005760.0   2040914       1

population  GISCO POP_2024 9,158,750  vs  new popTotal 9,215,956
count    2092.00
mean       -0.06
std         2.01
min       -16.48
25%        -1.12
50%        -0.12
75%         1.03
max        14.00

same code, different name: 2
lau_code_2026          LAU_NAME                name
        41346 St. Stefan-Afiesl   St. Stefan Afiesl
        41713           Lenzing Lenzing an der Ager


## 3. Write — common-schema dump

One CSV per country, same columns in all five `00_b*` notebooks, read by
`00_extract_base_data_eu.ipynb` §2c to attach the 2025 population to the LAU subset GeoJSON.

| column | content |
|---|---|
| `lau_code`, `gisco_id` | LAU code (text) and `<CC>_<lau_code>`, the subset key |
| `country`, `lau_name` | country code, national name |
| `pop_total`, `y_0_14`, `y_15_64`, `y_65plus` | persons (`pop.HARMONISED_BANDS`) |
| `band_method`, `ref_year`, `ref_date`, `source` | `exact`; reference year/date; dataset id |
| `native_id` | the source's own key |
| `replaces_lau` | workbook LAU codes a merged row covers (`;`-joined), else empty |
| `includes_native` | native units summed into the row (`;`-joined), else empty |

Skips if the file exists; set `REWRITE_DUMP = True` to overwrite. Read it back with `dtype=str`
for the code columns (HU codes have leading zeros).

In [13]:
REWRITE_DUMP = False
DUMP_COLS = ['lau_code', 'gisco_id', 'country', 'lau_name', 'pop_total', *pop.HARMONISED_BANDS,
             'band_method', 'ref_year', 'ref_date', 'source', 'native_id', 'replaces_lau', 'includes_native']
DUMP_FILE = OUT_DIR / f'lau_population_2025_{COUNTRY.lower()}.csv'

f = at_new_lau.reset_index(drop=True)
dump = pd.DataFrame({
    'lau_code': f['lau_code'].astype(str),
    'gisco_id': COUNTRY + '_' + f['lau_code'].astype(str),
    'country': COUNTRY,
    'lau_name': f['name'],
    'pop_total': f['popTotal'], 'y_0_14': f['popAge0to14'], 'y_15_64': f['popAge15to64'], 'y_65plus': f['popAge65over'],
    'band_method': 'exact',
    'ref_year': f['ref_year'], 'ref_date': f['ref_date'],
    'source': 'OGD_bevstandjbab2002_BevStand_2026',
    'native_id': f['id'].astype(str),
    'replaces_lau': f['replaces_lau'],
    'includes_native': f['includes_gkz'],
})[DUMP_COLS]

assert dump['lau_code'].is_unique and dump['gisco_id'].is_unique
assert (dump[pop.HARMONISED_BANDS].sum(axis=1) == dump['pop_total']).all(), 'bands do not add up'
assert dump[['pop_total', *pop.HARMONISED_BANDS]].notna().all().all()
replaced = {o for r in dump['replaces_lau'].dropna() for o in r.split(';')}
merged_new = set(dump.loc[dump['replaces_lau'].notna(), 'lau_code'])
assert (set(dump['lau_code']) - merged_new) | replaced == set(lau['lau_code']), 'dump does not cover the workbook LAUs'

if DUMP_FILE.exists() and not REWRITE_DUMP:
    print('[exists]', DUMP_FILE, '-- set REWRITE_DUMP = True to overwrite')
else:
    dump.to_csv(DUMP_FILE, index=False, encoding='utf-8')
    print('[wrote] ', DUMP_FILE)
back = pd.read_csv(DUMP_FILE, dtype={'lau_code': str, 'gisco_id': str, 'native_id': str})
assert list(back.columns) == DUMP_COLS and back['lau_code'].tolist() == dump['lau_code'].tolist(), 'file differs from dump'
print(f'{len(back):,} rows, population {back.pop_total.sum():,}; '
      f"merged rows {int(back.replaces_lau.notna().sum())}, aggregated rows {int(back.includes_native.notna().sum())}")
back.head(3)

[exists] data\eu\ref-national-statistical-offices\at\lau_population_2025_at.csv -- set REWRITE_DUMP = True to overwrite
2,092 rows, population 9,215,956; merged rows 1, aggregated rows 1


,lau_code,gisco_id,country,lau_name,pop_total,y_0_14,y_15_64,y_65plus,band_method,ref_year,ref_date,source,native_id,replaces_lau,includes_native
0,10101,AT_10101,AT,Eisenstadt,16396,2238,10582,3576,exact,2025,2026-01-01,OGD_bevstandjbab2002_BevStand_2026,10101,NaN,NaN
1,10201,AT_10201,AT,Rust,1918,212,1137,569,exact,2025,2026-01-01,OGD_bevstandjbab2002_BevStand_2026,10201,NaN,NaN
2,10301,AT_10301,AT,Breitenbrunn am Neusiedler See,1931,218,1072,641,exact,2025,2026-01-01,OGD_bevstandjbab2002_BevStand_2026,10301,NaN,NaN
